In [ ]:
import pandas as pd
import ast
import seaborn as sns
import matplotlib.pyplot as plt

result_names = [
    # "results_2025-08-28_13-41-37_gpt2-xl",
    # "results_2025-09-10_15-37-47_gpt2-medium"
    "causal_trace_Qwen-Qwen3-4B_2025-11-06_10-12-43"
]

for filename in result_names:
    df = pd.read_csv(f"{filename}.csv")

    display(df)

    df['clean'] = df['clean'].apply(ast.literal_eval)

    df['corrupted'] = df['corrupted'].apply(ast.literal_eval)
    df['restored'] = df['restored'].apply(ast.literal_eval)

    num_of_layers = len(df['restored'][0])

    df_expanded = df['restored'].apply(pd.Series)

    df_final = pd.concat([df.drop('restored', axis=1), df_expanded], axis=1)
    df_final["clean_token"] = df_final["clean"].apply(lambda x: x[0])
    df_final["clean"] = df_final["clean"].apply(lambda x: x[1])

    df_ff = df_final
    df_ff["corrupted"] = df_ff["corrupted"].apply(lambda x: x[1])
    
    for i in range(num_of_layers):
        df_ff[i] = df_ff[i].apply(lambda x: x[1])
        df_ff[i] = df_ff[i] - df_ff["corrupted"]

    df_preproc = df_ff
    df_preproc = df_preproc.drop('prompt_num', axis=1)
    df_preproc = df_preproc.drop('clean', axis=1)
    df_preproc = df_preproc.drop('corrupted', axis=1)
    df_p_g = df_preproc.groupby(["run_number"]).max(["restored_token"]).drop("restored_token", axis=1)

    display(df_p_g)

    sns.barplot(data=df_p_g)
    plt.show()


In [ ]:
from src.utils import load_pretrained
from omegaconf import DictConfig

cfg = DictConfig({
    "model": {
        "name": "Qwen/Qwen3-0.6B",
        "models_dir": "./models",
        "second_moment_dir": "./second_moment_stats",
        "device": "cpu",
        "save_to_local": True, # Keep this True until you specifically do not want the model cached locally
        "layer_name_template": "model.layers.{}.mlp.down_proj",
        "layer": 0,
        "corruption_noise_multiplier": 0.287875235080719,
    }
})
model, tok = load_pretrained(cfg)

W0 = model.model.layers[0].mlp.down_proj.weight.detach().numpy()
W1 = model.model.layers[1].mlp.down_proj.weight.detach().numpy()

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix0 = cosine_similarity(W0)
similarity_matrix1 = cosine_similarity(W1)
print("Pair-wise Cosine Similarity Matrix0 mean:\n", similarity_matrix0.mean())
print("Pair-wise Cosine Similarity Matrix1 mean:\n", similarity_matrix1.mean())
print("Pair-wise Cosine Similarity Matrix0 STD:\n", similarity_matrix0.std())
print("Pair-wise Cosine Similarity Matrix1 STD:\n", similarity_matrix1.std())

Pair-wise Cosine Similarity Matrix:
 [[ 1.0000023  -0.12125847 -0.00655    ...  0.06213947 -0.05979143
  -0.02637127]
 [-0.12125847  1.0000026  -0.08726873 ... -0.04747289  0.00587948
   0.00923493]
 [-0.00655    -0.08726873  1.0000006  ...  0.01225866 -0.00540599
  -0.02696775]
 ...
 [ 0.06213947 -0.04747289  0.01225866 ...  1.0000012   0.01252951
   0.03407587]
 [-0.05979143  0.00587948 -0.00540599 ...  0.01252951  1.0000021
  -0.00122684]
 [-0.02637127  0.00923493 -0.02696775 ...  0.03407587 -0.00122684
   1.0000012 ]]
Pair-wise Cosine Similarity Matrix STD:
 0.037465885
Pair-wise Cosine Similarity Matrix STD:
 0.037916586


In [3]:
import json
from typing import Optional, List
import collections
import numpy as np
from scipy.stats import hmean
from pprint import pprint
from pathlib import Path


RESULTS_DIR = Path("./evals")

def summarize(
    dir_name,
    runs: Optional[List],
    first_n_cases=None,
    get_uncompressed=False,
    abs_path=False,
):  # runs = None -> all runs
    summaries = []
    uncompressed = []

    # for run_dir in (RESULTS_DIR / dir_name if not abs_path else dir_name).iterdir():
    #     # Skip if we're not interested
    #     if runs is not None and all(run not in str(run_dir) for run in runs):
    #         continue

    #     # Iterate through all case files
    #     cur_sum = collections.defaultdict(lambda: [])
    #     files = list(run_dir.glob("case_*.json"))
    #     files.sort(key=lambda x: int(str(x).split("_")[-1].split(".")[0]))

        # for case_file in files:
    cur_sum = collections.defaultdict(lambda: [])
    for case_file in dir_name.iterdir():
        try:
            with open(case_file, "r") as f:
                data = json.load(f)
        except json.JSONDecodeError:
            print(f"Could not decode {case_file} due to format error; skipping.")

        case_id = data["case_id"]
        if first_n_cases is not None and case_id >= first_n_cases:
            break

        # cur_sum["time"].append(data["time"])

        for prefix in ["pre", "post"]:
            # Probability metrics for which new should be lower (better) than true
            for key in ["rewrite_prompts_probs", "paraphrase_prompts_probs"]:
                if prefix not in data or key not in data[prefix]:
                    continue


                sum_key_discrete = f"{prefix}_{key.split('_')[0]}_success"
                sum_key_cont = f"{prefix}_{key.split('_')[0]}_diff"

                cur_sum[sum_key_discrete].append(
                    np.mean(
                        [
                            x["target_true"] > x["target_new"]
                            for x in data[prefix][key]
                        ]
                    )
                )
                cur_sum[sum_key_cont].append(
                    np.mean(
                        [
                            np.exp(-x["target_new"]) - np.exp(-x["target_true"])
                            for x in data[prefix][key]
                        ]
                    )
                )

            # Probability metrics for which true should be lower (better) than new
            sum_key_discrete = f"{prefix}_neighborhood_success"
            sum_key_cont = f"{prefix}_neighborhood_diff"
            key = "neighborhood_prompts_probs"
            if prefix in data and key in data[prefix]:
                cur_sum[sum_key_discrete].append(
                    np.mean(
                        [
                            x["target_true"] < x["target_new"]
                            for x in data[prefix][key]
                        ]
                    )
                )
                cur_sum[sum_key_cont].append(
                    np.mean(
                        [
                            np.exp(-x["target_true"]) - np.exp(-x["target_new"])
                            for x in data[prefix][key]
                        ]
                    )
                )

            # zsRE evaluation metrics
            for key in ["rewrite", "paraphrase", "neighborhood"]:
                sum_key = f"{prefix}_{key}_acc"
                key = f"{key}_prompts_correct"

                if prefix not in data or key not in data[prefix]:
                    continue

                cur_sum[sum_key].append(np.mean(data[prefix][key]))

            # Generation metrics that can be directly averaged
            for key in ["ngram_entropy", "reference_score", "essence_score"]:
                if prefix in data and key in data[prefix]:
                    cur_sum[f"{prefix}_{key}"].append(data[prefix][key])

    num_items = len(cur_sum[next(iter(cur_sum.keys()))])
    metadata = {
        "run_dir": str(dir_name),
        "num_cases": num_items,
    }

    uncompressed.append(dict(cur_sum, **metadata))

    cur_sum = {k: (np.mean(v), np.std(v)) for k, v in cur_sum.items()}
    for prefix in ["pre", "post"]:
        for k_efficacy, k_generalization, k_specificity in [
            (
                f"{prefix}_rewrite_success",
                f"{prefix}_paraphrase_success",
                f"{prefix}_neighborhood_success",
            ),
            (
                f"{prefix}_rewrite_acc",
                f"{prefix}_paraphrase_acc",
                f"{prefix}_neighborhood_acc",
            ),
        ]:
            if k_generalization in cur_sum and k_specificity in cur_sum:
                cur_sum[f"{prefix}_score"] = (
                    hmean(
                        [
                            cur_sum[k_efficacy][0],
                            cur_sum[k_generalization][0],
                            cur_sum[k_specificity][0],
                        ]
                    ),
                    np.nan,
                )
                break

    for k, v in cur_sum.items():
        if all(exclude not in k for exclude in ["essence_score", "time"]):
            # Constant multiplication scales linearly with mean and stddev
            cur_sum[k] = tuple(np.around(z * 100, 2) for z in v)

    cur_sum.update(metadata)
    # pprint(cur_sum)
    summaries.append(cur_sum)

    return uncompressed if get_uncompressed else summaries

In [21]:
res = summarize(
    RESULTS_DIR,
    runs=None,
    abs_path=True,
)
stats_c = dict(res[0])
print(stats_c.keys())

dict_keys(['pre_rewrite_success', 'pre_rewrite_diff', 'pre_paraphrase_success', 'pre_paraphrase_diff', 'pre_neighborhood_success', 'pre_neighborhood_diff', 'post_rewrite_success', 'post_rewrite_diff', 'post_paraphrase_success', 'post_paraphrase_diff', 'post_neighborhood_success', 'post_neighborhood_diff', 'pre_score', 'post_score', 'run_dir', 'num_cases'])


In [10]:
res = summarize(
    RESULTS_DIR,
    runs=None,
    abs_path=True,
    get_uncompressed=True
)
stats = dict(res[0])
print(stats.keys())

dict_keys(['pre_rewrite_success', 'pre_rewrite_diff', 'pre_paraphrase_success', 'pre_paraphrase_diff', 'pre_neighborhood_success', 'pre_neighborhood_diff', 'post_rewrite_success', 'post_rewrite_diff', 'post_paraphrase_success', 'post_paraphrase_diff', 'post_neighborhood_success', 'post_neighborhood_diff', 'run_dir', 'num_cases'])


In [16]:
def efficacy(stats):
    return sum(stats["post_rewrite_success"])/len(stats["post_rewrite_success"])

efficacy(stats)

np.float64(0.9904544300583904)

In [18]:
def paraphrase(stats):
    return sum(stats["post_paraphrase_success"])/len(stats["post_paraphrase_success"])

paraphrase(stats)

np.float64(0.9794364051789795)

In [19]:
def specificity(stats):
    return sum(stats["post_neighborhood_success"])/len(stats["post_neighborhood_success"])

specificity(stats)

np.float64(0.7315816197003988)